# Pokémon Battle Prediction — FDS Kaggle Competition

## 1. Choose the exact source copy

Default: load the modified folder uploaded as a Kaggle Dataset (`fds-pokemon-source/fds-pokemon`), or set `FDS_POKEMON_SOURCE`/`LOCAL_SOURCE_DIR` to your local checkout. This does not require push access to the original repository. Alternatively choose `SOURCE_MODE = "fork"` and set your fork URL and revision. No automatic fallback to upstream occurs. The notebook prints the imported module path and saves hashes of the actual source, including uncommitted files. Restart the kernel when changing source. See `docs/VALIDATION_PROTOCOLS.md` for packaging instructions.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Explicit choice: use the modified local/uploaded source, or your own fork.
SOURCE_MODE = "local"  # "local" or "fork"
LOCAL_SOURCE_DIR = Path(os.environ.get("FDS_POKEMON_SOURCE", "/kaggle/input/fds-pokemon-source/fds-pokemon"))
REPO_URL = "https://github.com/<your-user>/fds-pokemon.git"
REPO_REVISION = "main"  # Prefer a commit hash for a reproducible fork checkout.
FORK_CHECKOUT = Path("/kaggle/working/fds-pokemon-fork")
RUN_STRENGTH_RESEARCH = False

if any(name == "pokemon" or name.startswith("pokemon.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before loading or changing the source copy.")
if SOURCE_MODE == "local":
    REPO_DIR = LOCAL_SOURCE_DIR.resolve()
elif SOURCE_MODE == "fork":
    if "<your-user>" in REPO_URL:
        raise ValueError("Set REPO_URL to your fork, containing these modifications.")
    REPO_DIR = FORK_CHECKOUT
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    remote = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    if remote != REPO_URL:
        raise RuntimeError("Existing checkout belongs to another remote; choose a different FORK_CHECKOUT.")
    dirty = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip()
    if dirty:
        raise RuntimeError("Fork checkout has local changes. Use local mode or a fresh directory.")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REVISION], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
else:
    raise ValueError("SOURCE_MODE must be local or fork")
for filename in ("pokemon/main.py", "pokemon/protocols.py", "pokemon/validation.py", "requirements.txt"):
    if not (REPO_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing modified source: {REPO_DIR / filename}")
REPO_COMMIT = "exported-source-without-git"
if (REPO_DIR / ".git").exists():
    REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Source directory:", REPO_DIR)
print("Repository commit:", REPO_COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(REPO_DIR))
import pokemon.main
if not Path(pokemon.main.__file__).resolve().is_relative_to(REPO_DIR.resolve()):
    raise RuntimeError("Imported pokemon.main from an unexpected source copy.")
print("Executing:", pokemon.main.__file__)


## 2. Locate competition data and build features

The feature builder only uses the available battle observations, the supplied type chart, and species statistics already present in the official JSONL files.


In [ ]:
import hashlib
import importlib
import importlib.metadata
import json
import platform
from pathlib import Path
from pokemon.main import build_all, cross_validate, make_submission, make_model, MODEL_NAMES

train_files = list(Path("/kaggle/input").rglob("train.jsonl"))
if len(train_files) != 1:
    raise FileNotFoundError(f"Expected exactly one train.jsonl, found {len(train_files)}")

DATA_DIR = train_files[0].parent
for filename in ("test.jsonl", "type_chart_gen2.csv"):
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing {filename} in {DATA_DIR}")

def file_sha256(path):
    checksum = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            checksum.update(chunk)
    return checksum.hexdigest()

packages = {
    "numpy": "numpy", "pandas": "pandas", "scikit-learn": "sklearn",
    "lightgbm": "lightgbm", "xgboost": "xgboost", "catboost": "catboost",
}
manifest = {
    "repo_commit": REPO_COMMIT,
    "python": sys.version,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "versions": {
        name: importlib.import_module(module).__version__
        for name, module in packages.items()
    },
    "installed_versions": {name: importlib.metadata.version(name) for name in packages},
    "input_sha256": {
        name: file_sha256(DATA_DIR / name)
        for name in ("train.jsonl", "test.jsonl", "type_chart_gen2.csv")
    },
    "model_parameters": json.loads(
        json.dumps({name: make_model(name).get_params() for name in MODEL_NAMES}),
        parse_constant=str,
    ),
    "validation": {"n_splits": 5, "shuffle": True, "random_state": 42, "threshold": 0.5},
}
print("Loaded library versions:", json.dumps(manifest["versions"], sort_keys=True))
if manifest["versions"] != manifest["installed_versions"]:
    raise RuntimeError("Loaded and installed library versions differ. Restart the kernel after installing dependencies.")

features = build_all(DATA_DIR)
print("Training features:", features["A"].shape)
print("Test features:", features["B"].shape)
manifest["feature_columns"] = list(features["A"].columns)
manifest["feature_schema_sha256"] = hashlib.sha256(
    json.dumps(manifest["feature_columns"]).encode("utf-8"),
).hexdigest()
MANIFEST_PATH = Path("/kaggle/working/run_manifest.json")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print("Run manifest:", MANIFEST_PATH)


## 3. Validation

We use stratified five-fold cross-validation with **nested target encoding**. In every fold, validation labels are excluded from training and encoding statistics. The values printed below are the measured OOF accuracies for this run, not Kaggle test scores.

The latest supplied Kaggle execution (26 October 2026) measured **69.33% LightGBM**, **69.64% LightGBM/XGBoost**, and **69.72% for the 45/45/10 blend**, after selecting XGBoost 3.1.3 and CatBoost 1.2.8. These are reference observations, **not guaranteed results**. The earlier 69.79% ensemble score is historical. Direct dependencies are now pinned in `requirements.txt`; this run's actual library versions, Git commit, and input checksums are saved in `run_manifest.json`. Always rely on the actual output printed by `cross_validate(features)`.

Diagnostics are saved under `/kaggle/working/validation/original`: `validation_predictions.csv` includes source row, battle ID, fold, outcome, and each model's probability, binary decision and correctness. `validation_report.json` includes fold scores/counts, paired error changes versus LightGBM, the training-only majority baseline, split/model/encoder seeds, code/data hashes and team-overlap diagnostics. Duplicate observations/IDs are kept together; if any exist, the original random assignments are recorded for audit but protected groups are used for fitting. Threshold ties predict 0. Set `REFERENCE_OOF` to compare with a previous file on the same battles/folds.


In [ ]:
# Diagnostics are kept separate from the Kaggle submission CSVs.
VALIDATION_DIR = Path("/kaggle/working/validation/original")
REFERENCE_OOF = None  # A previous validation_predictions.csv, if available.
cv_report, cv_oof = cross_validate(
    features, output_dir=VALIDATION_DIR, reference_oof=REFERENCE_OOF,
    return_details=True, outer_seed=42, model_seed=42, te_seed=0,
)
scores = {name: metrics["accuracy"] for name, metrics in cv_report["models"].items()}
print(json.dumps(cv_report, indent=2))
manifest["baseline_cv"] = scores
manifest["baseline_cv_report"] = cv_report
manifest["source_provenance"] = cv_report["provenance"]
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")


### Optional validation protocols — run one intervention at a time

Disabled by default. `repeated` varies only the outer seed (42, 123, 2026); repeated scores are dependent and no best seed is selected. `grouped` tests P1 compositions excluded from P1 training, while reporting fully revealed P2 overlap. `preparation` compares global unlabeled catalogs with transformations fitted inside each training fold, using the same splits/models. `nested` selects among the three fixed blends using three inner folds and evaluates only on the outer fold. Encoder cross-fitting alone is not independent model selection. Previous experiments have already influenced the finalist list; new splits do not erase that history. These protocols do not change the submission models below. See `docs/VALIDATION_PROTOCOLS.md` for commands and interpretation.


In [ ]:
from pokemon.main import repeat_validation, compare_preparation

OPTIONAL_PROTOCOL = None  # None, "repeated", "grouped", "preparation", or "nested"
PROTOCOL_ROOT = Path("/kaggle/working/validation")
if OPTIONAL_PROTOCOL == "repeated":
    protocol_result = repeat_validation(features, PROTOCOL_ROOT / "repeated")
elif OPTIONAL_PROTOCOL == "grouped":
    protocol_result, _ = cross_validate(features, split_mode="p1_grouped",
                                       output_dir=PROTOCOL_ROOT / "grouped", return_details=True)
elif OPTIONAL_PROTOCOL == "preparation":
    protocol_result = compare_preparation(features, PROTOCOL_ROOT / "preparation")
elif OPTIONAL_PROTOCOL == "nested":
    protocol_result, _ = cross_validate(features, nested=True, preparation="fold",
                                       output_dir=PROTOCOL_ROOT / "nested_fold", return_details=True)
elif OPTIONAL_PROTOCOL is not None:
    raise ValueError("Unknown OPTIONAL_PROTOCOL")
if OPTIONAL_PROTOCOL is not None:
    print(json.dumps(protocol_result, indent=2))


## 4. Submission 1 — LightGBM

A strongly regularized tree ensemble using team, resource, tactical, and cross-fitted target-encoding features. Reference Kaggle five-fold accuracy: **69.33%**. The current run's accuracy is printed below before generating the CSV.


In [ ]:
print(f"Measured OOF accuracy: {scores['sub1_lgbm']:.2%}")
submission_1 = make_submission(features, "sub1_lgbm", "/kaggle/working")
print(submission_1)


## 5. Submission 2 — LightGBM + XGBoost

An **equal 50/50 average** of the two probability predictions. XGBoost provides a different boosting implementation and regularization regime. Reference Kaggle five-fold accuracy: **69.64%**. The current run's accuracy is printed below.


In [ ]:
print(f"Measured OOF accuracy: {scores['sub2_lgb_xgb']:.2%}")
submission_2 = make_submission(features, "sub2_lgb_xgb", "/kaggle/working")
print(submission_2)


## 6. Submission 3 — LightGBM + XGBoost + CatBoost

A **45% / 45% / 10% probability blend**, previously selected using labeled-training cross-validation. Reference Kaggle five-fold accuracy: **69.72%**. The current run's accuracy is printed below; averaging models does not guarantee improvement in every environment.


In [ ]:
print(f"Measured OOF accuracy: {scores['sub3_lgb_xgb_cat']:.2%}")
submission_3 = make_submission(features, "sub3_lgb_xgb_cat", "/kaggle/working")
print(submission_3)


## 7. Validate submission format

All predictions must have the same `battle_id` values and order as the official competition template, with `player_won` restricted to 0 or 1.


In [ ]:
import pandas as pd

sample_files = list(Path("/kaggle/input").rglob("sample_submission.csv"))
if len(sample_files) != 1:
    raise FileNotFoundError("Expected exactly one sample_submission.csv")

sample = pd.read_csv(sample_files[0])
for output in (submission_1, submission_2, submission_3):
    submission = pd.read_csv(output)
    assert submission.columns.tolist() == sample.columns.tolist()
    assert submission["battle_id"].equals(sample["battle_id"])
    assert submission["player_won"].isin([0, 1]).all()
    print(f"Validated {Path(output).name}: {len(submission)} predictions")


## 8. Active-strength candidate: validate in this Kaggle environment

The 637-column baseline above does not include the newer 42 interactions between active HP, stats, stat stages and status conditions. This experiment evaluates XGBoost with those features and seed 42, using the same five folds and nested encodings. It compares the candidate against the original ensemble **in this environment**, records OOF predictions and generates separate candidate CSVs.

The individual candidate previously scored 69.85% locally; that result is not a promised Kaggle score and does not establish 70% accuracy. `RUN_STRENGTH_RESEARCH = False` is the default so the notebook produces only the three documented submissions. Set it to `True` in the setup cell to enable this additional experiment. Candidate files require a separate selection decision; producing them does not select additional Kaggle submissions.


In [ ]:
if RUN_STRENGTH_RESEARCH:
    from strength_experiment import validate_strength, generate_candidates
    from pokemon.strength_features import build_strength_features

    strength_width = features["A"].shape[1] + build_strength_features(features["A"]).shape[1]
    print("Strength input columns (before target encodings):", strength_width)
    strength_results, strength_oof = validate_strength(features, seeds=(42,))
    strength_results["train_sha256"] = manifest["input_sha256"]["train.jsonl"]
    strength_results["repo_commit"] = REPO_COMMIT
    strength_results["run_manifest"] = str(MANIFEST_PATH)
    oof_path = Path("/kaggle/working/strength_oof.csv.gz")
    strength_oof.to_csv(oof_path, index=False)
    strength_results["oof_file"] = oof_path.name

    for name, result in strength_results["candidates"].items():
        print(f"{name}: {result['accuracy']:.2%}")
    candidate_paths = generate_candidates(
        features, "/kaggle/working/strength_candidates", seeds=(42,),
    )
    for output in candidate_paths.values():
        candidate = pd.read_csv(output)
        assert candidate.columns.tolist() == sample.columns.tolist()
        assert candidate["battle_id"].equals(sample["battle_id"])
        assert candidate["player_won"].isin([0, 1]).all()
        print(f"Validated {output}: {len(candidate)} predictions")

    strength_results["submission_files"] = candidate_paths
    Path("/kaggle/working/strength_results.json").write_text(
        json.dumps(strength_results, indent=2) + "\n", encoding="utf-8",
    )
    manifest["strength_cv"] = strength_results["candidates"]
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
else:
    print("Strength research disabled; set RUN_STRENGTH_RESEARCH = True in the setup cell to enable it.")
